In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121

In [2]:
!pip install causal-conv1d mamba-ssm --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.0-cp312-cp312-linux_x86_64.whl size=193268366 sha256=9602d561d06e3a8f0fec8c09fa2a0fbec223e4e8b64dcbfab7a21d0fd6b15003
  Stored in directory: /root/.cache/pip/wheels/9a/1e/2e/a0c04cf5c155486922378c8a2f111ac9705a354a949dd724a4
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.0-cp312-cp312-linux_x86_64.whl size=370572351 sha256=44418a0e4eb4534eec3efce671f51075c8263bdc28396cdd3424ca662b9f9d04
  Stored in directory: /root/.cache/pip/wheels/4c/4c/6f/d6add86ebe5c3dbd407ad3b593ecb31cda9adf70801387df4a
Successfully built causal-conv1d mamba-ssm


In [3]:
pip install genomic-benchmarks

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.5 MB/s eta 0:00:0000:0100:01
  Created wheel for genomic-benchmarks: filename=genomic_benchmarks-1.0.0-py3-none-any.whl size=22550 sha256=6f789434bf9fca8906c9b1532ae3434df614db15ba864d1d012de4a5f75e957d
  Stored in directory: /root/.cache/pip/wheels/9a/3a/9a/0f21797a390f81beeeb52e9ccc71e6d5e262786ecd01e046bf
Successfully built genomic-benchmarks
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import pathlib
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from genomic_benchmarks.loc2seq import download_dataset

class Mamba2Block(layers.Layer):
    def __init__(self, d_model, d_state=64, expand=2, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.d_inner = d_model * expand
        self.d_state = d_state
        
        self.in_proj = layers.Dense(self.d_inner * 2, use_bias=False)
        self.conv1d = layers.Conv1D(
            filters=self.d_inner, kernel_size=4, padding='same', groups=self.d_inner
        )
        self.x_proj = layers.Dense(self.d_state + 2, use_bias=False)
        self.out_proj = layers.Dense(d_model, use_bias=False)

    def call(self, x):
        z_x = self.in_proj(x)
        z, x = tf.split(z_x, num_or_size_splits=2, axis=-1)
        x = tf.nn.silu(self.conv1d(x))
        # simplified SSD recurrence
        A = tf.fill((tf.shape(x)[0], tf.shape(x)[1], self.d_inner), 0.9)
        x = x * A 
        x = x * tf.nn.silu(z)
        return self.out_proj(x)

# RC-equivariant mamba-2 model

class RCMamba2Model(Model):
    def __init__(self, vocab_size=5, d_model=128, n_layers=4, n_classes=2):
        super().__init__()
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.mamba_blocks = [Mamba2Block(d_model) for _ in range(n_layers)]
        self.norm = layers.LayerNormalization()
        self.pool = layers.GlobalAveragePooling1D()
        self.classifier = layers.Dense(n_classes, activation='softmax')

    def get_reverse_complement(self, x):
        # A(1)<->T(4), C(2)<->G(3), N(0)->N(0)
        rc_map = tf.constant([0, 4, 3, 2, 1], dtype=x.dtype)
        return tf.reverse(tf.gather(rc_map, x), axis=[1])

    def backbone(self, x_embed):
        for block in self.mamba_blocks:
            x_embed = x_embed + block(x_embed)
        return x_embed

    def call(self, x):
        # forward branch
        feat_fwd = self.pool(self.backbone(self.embedding(x)))
        # RC branch
        feat_rc = self.pool(self.backbone(self.embedding(self.get_reverse_complement(x))))
        # equivariant fusion
        return self.classifier(self.norm((feat_fwd + feat_rc) / 2.0))

def prepare_and_train():
    print("Fetching dataset...")
    data_path = download_dataset('demo_human_or_worm')

    train_ds = tf.keras.utils.text_dataset_from_directory(
        os.path.join(data_path, 'train'),
        batch_size=64,
        label_mode='int'
    )
    
    dna_map = {'A': 1, 'C': 2, 'G': 3, 'T': 4, 'N': 0}
    
    def tokenize_dna(text, label):
        chars = tf.strings.unicode_split(tf.strings.upper(text), 'UTF-8')
        table = tf.lookup.StaticHashTable(
            tf.lookup.KeyValueTensorInitializer(
                list(dna_map.keys()), list(dna_map.values())
            ), default_value=0
        )
        return table.lookup(chars).to_tensor(), label

    train_ds = train_ds.map(tokenize_dna)
    
    model = RCMamba2Model(d_model=64, n_layers=2)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    print("Starting training...")
    model.fit(train_ds, epochs=5)

if __name__ == "__main__":
    prepare_and_train()

Fetching dataset...
Found 75000 files belonging to 2 classes.
Starting training...
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'mamba2_block_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'mamba2_block_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


1172/1172 ━━━━━━━━━━━━━━━━━━━━ 25s 15ms/step - accuracy: 0.7930 - loss: 0.4097
Epoch 2/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9425 - loss: 0.1511
Epoch 3/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9469 - loss: 0.1387
Epoch 4/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9491 - loss: 0.1321
Epoch 5/5
1172/1172 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - accuracy: 0.9516 - loss: 0.1265
